# 3. Otimização — bateria de métodos

Mesma PNL, três métodos, **cinco partidas** compartilhadas (semente 0):

1. reta no espaço das juntas (inviável: atravessa o disco);
2. desvio pela esquerda, viável;
3–5. ruído gaussiano sobre o desvio, reamostrado até ficar viável.

| Método | Implementação |
|--------|----------------|
| Pontos interiores | `trust-constr` (barreira) com Jacobiana analítica, polido por SQP até um ponto KKT |
| Pattern search | COBYLA (Powell), sem derivadas; para quando o raio da região de confiança atinge o mínimo |
| Newton | SQP (`SLSQP`): Newton no sistema KKT com Hessiana quasi-Newton do Lagrangiano |

Sucesso = ponto **viável** e parada de ótimo local (`gtol` / `xtol` / KKT / raio mínimo), não o teto de iterações.

Se `T2/saida/bateria.pkl` existe **e** foi gravado com o mesmo \(K\), o
notebook o recarrega. Senão a bateria é recalculada.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
for cand in [here, here.parent, here / "T2", here.parent / "T2"]:
    if (cand / "src" / "parametros.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
from src.ambiente import preparar
T2 = preparar()
print("T2 =", T2)


In [ ]:
from src.experiencia import carregar_ou_executar
from src.parametros import P

payload = carregar_ou_executar()
registros = payload["registros"]
resumo = payload["resumo"]
print("K =", payload.get("K", P.K), "n_z =", payload.get("n_z"))
print("erro da Jacobiana eq/ineq:", payload["jac_err_eq"], payload["jac_err_ineq"])


In [ ]:
print(f"{'método':20s} {'chute':18s} {'tf':>8s} {'conv':>5s} {'viável':>7s} {'eq':>10s} {'ineq':>10s} {'iter':>6s} {'CPU':>8s}")
for r in registros:
    print(f"{r['metodo']:20s} {r['chute']:18s} {r['tf']:8.4f} {str(r['convergiu']):>5s} {str(r['viavel']):>7s} "
          f"{r['violacao_eq']:10.2e} {r['violacao_ineq']:10.2e} {r['niter']:6d} {r['cpu_s']:8.3f}")


In [ ]:
print(f"{'método':20s} {'ok':>6s} {'tf min':>8s} {'tf méd':>8s} {'tf máx':>8s} {'std':>8s} {'CPU méd':>8s}")
for m, s in resumo.items():
    print(f"{m:20s} {s['n_sucesso']}/{s['n']:<3d} {s['tf_min']:8.4f} {s['tf_media']:8.4f} "
          f"{s['tf_max']:8.4f} {s['tf_std']:8.4f} {s['cpu_media']:8.3f}")
melhor = payload["melhor"]
print(f"\nmelhor (viável e convergido): {melhor['metodo']} / {melhor['chute']}  tf={melhor['tf']:.4f} s")
